# Transparent Adaptive Taxation: Reproducible Eight-Round Game

This self-contained notebook reconstructs the deterministic taxpayer-authority game used in Yichen Shen's COMSCI/ECON 206 PS1. It requires no API key, network call, or third-party package. Outputs are programmed demonstrations, not evidence about real taxpayers.

In [ ]:
from dataclasses import dataclass, field

MAX_ROUNDS, INITIAL_TRUST = 8, 50
RULES = {('C','T'):(3,3,10), ('C','O'):(0,4,-15), ('A','T'):(4,0,-10), ('A','O'):(1,1,-10)}
PRESETS = {
    'transparent_cooperation': [('C','T')]*8,
    'opaque_breakdown': [('C','O')]+[('A','O')]*7,
    'mixed_recovery': [('C','T'),('C','T'),('A','T'),('A','O')]+[('C','T')]*4,
}

@dataclass
class State:
    round: int = 1
    trust: int = INITIAL_TRUST
    taxpayer_payoff: int = 0
    authority_payoff: int = 0
    log: list = field(default_factory=list)

def run(path, initial_trust=INITIAL_TRUST):
    if not 0 <= initial_trust <= 100: raise ValueError('initial trust must be in [0,100]')
    if len(path) > MAX_ROUNDS: raise ValueError('at most eight rounds')
    s = State(trust=initial_trust)
    for taxpayer, authority in path:
        if (taxpayer, authority) not in RULES: raise ValueError('invalid action pair')
        pt, pa, dt = RULES[(taxpayer, authority)]
        before = s.trust
        s.taxpayer_payoff += pt; s.authority_payoff += pa
        s.trust = min(100, max(0, s.trust + dt))
        s.log.append({'round':s.round, 'actions':f'{taxpayer}/{authority}', 'payoffs':[pt,pa], 'trust_before':before, 'trust_after':s.trust})
        s.round += 1
    return s


## One-shot economic benchmark
Avoid is strictly dominant for the taxpayer and Opaque is strictly dominant for the authority; therefore A/O is the unique one-shot Nash equilibrium. C/T yields higher payoffs to both players.

In [ ]:
benchmark = {
    'taxpayer_avoid_strictly_dominant': RULES[('A','T')][0] > RULES[('C','T')][0] and RULES[('A','O')][0] > RULES[('C','O')][0],
    'authority_opaque_strictly_dominant': RULES[('C','O')][1] > RULES[('C','T')][1] and RULES[('A','O')][1] > RULES[('A','T')][1],
    'unique_one_shot_nash': 'A/O', 'pareto_superior_profile': 'C/T'
}
benchmark

## Reproduce the three submitted paths

In [ ]:
results = {}
for name, path in PRESETS.items():
    s = run(path)
    results[name] = {'taxpayer_payoff':s.taxpayer_payoff, 'authority_payoff':s.authority_payoff, 'final_trust':s.trust, 'rounds':len(s.log)}
results

In [ ]:
assert tuple(results['transparent_cooperation'][k] for k in ('taxpayer_payoff','authority_payoff','final_trust')) == (24,24,100)
assert tuple(results['opaque_breakdown'][k] for k in ('taxpayer_payoff','authority_payoff','final_trust')) == (7,11,0)
assert tuple(results['mixed_recovery'][k] for k in ('taxpayer_payoff','authority_payoff','final_trust')) == (23,19,90)
assert all(value['rounds'] == 8 for value in results.values())
print('All submitted paths and complete logs verified.')

## One bounded sensitivity check
The next cell changes only initial trust for the same mixed path. It checks clipping and path dependence in the programmed score; it is not a behavioral experiment.

In [ ]:
sensitivity = {start: run(PRESETS['mixed_recovery'], initial_trust=start).trust for start in (20,50,80)}
sensitivity

## Interpretation boundary
The notebook verifies arithmetic, trust bounds, and event-log completeness. It does not show that transparency causes compliance. The proposed next study independently varies explanation transparency and continuation probability to distinguish institutional trust from strategic learning.